In [1]:
from tests.adapters import run_get_response_log_probs, run_tokenize_prompt_and_output, run_sft_microbatch_train_step
from torch import Tensor
from tqdm import tqdm
import torch
import numpy as np
import random
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
device = torch.device("cuda")

num_epochs = 1
batch_size = 8
gradient_accumulation_steps = 4

In [3]:
model = AutoModelForCausalLM.from_pretrained(
    'sft_model',
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2'
).to(device)
tokenizer = AutoTokenizer.from_pretrained('sft_model')

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


In [4]:
df = pd.read_parquet("expert_iteration_training_single.parquet")
prompts = df['prompt'].tolist()
responses = df['response'].tolist()
tokenized = run_tokenize_prompt_and_output(prompts, responses, tokenizer)
input_ids = tokenized['input_ids'].long().to(device)
labels = tokenized['labels'].long().to(device)
response_mask = tokenized['response_mask'].to(device)

In [5]:
from torch.utils.data import DataLoader, Dataset

class MathSFTDataset(Dataset):
    def __init__(self, input_ids, labels, masks):
        self.input_ids = input_ids
        self.labels = labels
        self.masks = masks

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        input_id = self.input_ids[index]
        label = self.labels[index]
        mask = self.masks[index]
        return input_id, label, mask

train_loader = DataLoader(
    dataset=MathSFTDataset(input_ids, labels, response_mask),
    batch_size=batch_size
)


In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)


In [7]:
global_step = -1

for epoch in range(num_epochs):
    model.train()

    for idx, (input_batch, label_batch, mask) in tqdm(enumerate(train_loader)):
        log_probs = run_get_response_log_probs(model, input_batch, label_batch, False)['log_probs']
        loss, _ = run_sft_microbatch_train_step(log_probs, mask, gradient_accumulation_steps)
        global_step += 1
        
        if (idx + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            
        print(f"Step {global_step:06d}: Train loss: {loss.cpu().item()}")


1it [00:27, 27.04s/it]

Step 000000: Train loss: 109.1303939819336


2it [00:27, 11.37s/it]

Step 000001: Train loss: 136.16293334960938


3it [00:27,  6.36s/it]

...

Step 000549: Train loss: 159.19749450683594


551it [04:07,  2.50it/s]

Step 000550: Train loss: 239.61944580078125


552it [04:08,  2.22it/s]

Step 000551: Train loss: 112.76799011230469


In [8]:
import os

output_dir = "sft_model_ei_single"
os.makedirs(output_dir, exist_ok=True)

print("saving the model and tokenizer...")
model.save_pretrained(save_directory=output_dir)
tokenizer.save_pretrained(save_directory=output_dir)

saving the model and tokenizer...


('sft_model_ei_single/tokenizer_config.json',
 'sft_model_ei_single/special_tokens_map.json',
 'sft_model_ei_single/chat_template.jinja',
 'sft_model_ei_single/vocab.json',
 'sft_model_ei_single/merges.txt',
 'sft_model_ei_single/added_tokens.json',
 'sft_model_ei_single/tokenizer.json')